In [ ]:
# load common model definitions

%run 04_models_ABC.ipynb


In [ ]:
# settings

CAPACITY_RESULTS_DIR = Path("results") / "capacity"
CAPACITY_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
CAPACITY_EPOCHS = 500
CAPACITY_LEARNING_RATE = 1e-3

WIDTHS = [
    (4, 2),
    (32, 16),
    (64, 32),
    (128, 64),
    (256, 128),
    (512, 256),
]

DROPOUT_RATES = [
    0.0,
    0.1,
    0.2,
    0.3,
    0.5,
]


In [ ]:
# load the modelling data

scada = pd.read_parquet(
    SCADA_FILE
).copy()

static = pd.read_parquet(
    STATIC_FILE
).copy()

farm_data, turbine_ids = build_modelling_data(
    scada
)

train_data, validation_data, _ = chronological_split(
    farm_data
)

feature_columns = [
    "global_ws",
    "wd_sin",
    "wd_cos",
]

wind_speed_columns = [
    f"ws_t{turbine_id}"
    for turbine_id in turbine_ids
]

turbine_power_columns = [
    f"power_t{turbine_id}"
    for turbine_id in turbine_ids
]

print(f"training rows: {len(train_data):,}")
print(f"validation rows: {len(validation_data):,}")


In [ ]:
# standardise inputs from the training period only

X_train_raw = train_data[
    feature_columns
].to_numpy()

X_validation_raw = validation_data[
    feature_columns
].to_numpy()

X_mean, X_std = fit_standardisation(
    X_train_raw
)

X_train = standardise(
    X_train_raw,
    X_mean,
    X_std,
)

X_validation = standardise(
    X_validation_raw,
    X_mean,
    X_std,
)


In [ ]:
# model with optional dropout

def build_capacity_network(
    input_size,
    output_size,
    hidden_layers,
    dropout=0.0,
):
    layers = []
    previous_size = input_size

    for hidden_size in hidden_layers:
        layers.append(
            nn.Linear(
                previous_size,
                hidden_size,
            )
        )

        layers.append(
            nn.ReLU()
        )

        if dropout > 0:
            layers.append(
                nn.Dropout(dropout)
            )

        previous_size = hidden_size

    layers.append(
        nn.Linear(
            previous_size,
            output_size,
        )
    )

    return nn.Sequential(*layers)


In [ ]:
# fixed-epoch training used for the capacity comparison

def train_fixed_epochs(
    model,
    X_train,
    y_train,
    epochs,
    learning_rate,
    seed,
):
    set_seed(seed)

    X_train_tensor = torch.tensor(
        X_train,
        dtype=torch.float32,
    )

    y_train_tensor = torch.tensor(
        y_train,
        dtype=torch.float32,
    )

    loss_function = nn.HuberLoss(
        delta=1.0
    )

    optimiser = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    loader = DataLoader(
        TensorDataset(
            X_train_tensor,
            y_train_tensor,
        ),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
    )

    for _ in range(epochs):
        model.train()

        for X_batch, y_batch in loader:
            optimiser.zero_grad()

            prediction = model(X_batch)

            loss = loss_function(
                prediction,
                y_batch,
            )

            loss.backward()
            optimiser.step()

    model.eval()

    return model


In [ ]:
# evaluate one architecture for models A, B and C

def evaluate_capacity(
    hidden_layers,
    seed=SEED,
):
    results = {}

    X_train_tensor = torch.tensor(
        X_train,
        dtype=torch.float32,
    )

    X_validation_tensor = torch.tensor(
        X_validation,
        dtype=torch.float32,
    )

    # model A

    y_A_train_raw = train_data[
        ["total_power"]
    ].to_numpy()

    y_A_validation_raw = validation_data[
        ["total_power"]
    ].to_numpy()

    y_A_mean, y_A_std = fit_standardisation(
        y_A_train_raw
    )

    y_A_train = standardise(
        y_A_train_raw,
        y_A_mean,
        y_A_std,
    )

    model_A = build_capacity_network(
        input_size=len(feature_columns),
        output_size=1,
        hidden_layers=hidden_layers,
    )

    model_A = train_fixed_epochs(
        model=model_A,
        X_train=X_train,
        y_train=y_A_train,
        epochs=CAPACITY_EPOCHS,
        learning_rate=CAPACITY_LEARNING_RATE,
        seed=seed,
    )

    with torch.no_grad():
        pred_A_train = (
            model_A(X_train_tensor).numpy()
            * y_A_std
            + y_A_mean
        ).reshape(-1)

        pred_A_validation = (
            model_A(X_validation_tensor).numpy()
            * y_A_std
            + y_A_mean
        ).reshape(-1)

    results["A_train_kW"] = mean_absolute_error(
        y_A_train_raw.reshape(-1),
        pred_A_train,
    )

    results["A_validation_kW"] = mean_absolute_error(
        y_A_validation_raw.reshape(-1),
        pred_A_validation,
    )

    # model B

    y_B_train_raw = train_data[
        wind_speed_columns
    ].to_numpy()

    y_B_validation_raw = validation_data[
        wind_speed_columns
    ].to_numpy()

    y_B_mean, y_B_std = fit_standardisation(
        y_B_train_raw
    )

    y_B_train = standardise(
        y_B_train_raw,
        y_B_mean,
        y_B_std,
    )

    model_B = build_capacity_network(
        input_size=len(feature_columns),
        output_size=len(turbine_ids),
        hidden_layers=hidden_layers,
    )

    model_B = train_fixed_epochs(
        model=model_B,
        X_train=X_train,
        y_train=y_B_train,
        epochs=CAPACITY_EPOCHS,
        learning_rate=CAPACITY_LEARNING_RATE,
        seed=seed,
    )

    with torch.no_grad():
        pred_B_train = (
            model_B(X_train_tensor).numpy()
            * y_B_std
            + y_B_mean
        )

        pred_B_validation = (
            model_B(X_validation_tensor).numpy()
            * y_B_std
            + y_B_mean
        )

    results["B_train_ms"] = mean_absolute_error(
        y_B_train_raw.reshape(-1),
        pred_B_train.reshape(-1),
    )

    results["B_validation_ms"] = mean_absolute_error(
        y_B_validation_raw.reshape(-1),
        pred_B_validation.reshape(-1),
    )

    # model C

    y_C_train_raw = train_data[
        turbine_power_columns
    ].to_numpy()

    y_C_validation_raw = validation_data[
        turbine_power_columns
    ].to_numpy()

    y_C_mean, y_C_std = fit_standardisation(
        y_C_train_raw
    )

    y_C_train = standardise(
        y_C_train_raw,
        y_C_mean,
        y_C_std,
    )

    model_C = build_capacity_network(
        input_size=len(feature_columns),
        output_size=len(turbine_ids),
        hidden_layers=hidden_layers,
    )

    model_C = train_fixed_epochs(
        model=model_C,
        X_train=X_train,
        y_train=y_C_train,
        epochs=CAPACITY_EPOCHS,
        learning_rate=CAPACITY_LEARNING_RATE,
        seed=seed,
    )

    with torch.no_grad():
        pred_C_train_turbines = (
            model_C(X_train_tensor).numpy()
            * y_C_std
            + y_C_mean
        )

        pred_C_validation_turbines = (
            model_C(X_validation_tensor).numpy()
            * y_C_std
            + y_C_mean
        )

    pred_C_train_farm = pred_C_train_turbines.sum(
        axis=1
    )

    pred_C_validation_farm = (
        pred_C_validation_turbines.sum(axis=1)
    )

    results["C_train_kW"] = mean_absolute_error(
        train_data["total_power"].to_numpy(),
        pred_C_train_farm,
    )

    results["C_validation_kW"] = mean_absolute_error(
        validation_data["total_power"].to_numpy(),
        pred_C_validation_farm,
    )

    results["A_parameters"] = sum(
        parameter.numel()
        for parameter in model_A.parameters()
    )

    results["B_parameters"] = sum(
        parameter.numel()
        for parameter in model_B.parameters()
    )

    results["C_parameters"] = sum(
        parameter.numel()
        for parameter in model_C.parameters()
    )

    return results


In [ ]:
# run the network-capacity sweep

capacity_rows = []

for hidden_layers in WIDTHS:
    result = evaluate_capacity(
        hidden_layers=hidden_layers,
        seed=SEED,
    )

    capacity_rows.append(
        {
            "hidden_layers": str(hidden_layers),
            **result,
        }
    )

capacity_results = pd.DataFrame(
    capacity_rows
)

capacity_results.to_csv(
    CAPACITY_RESULTS_DIR / "capacity_results.csv",
    index=False,
)

print(
    capacity_results[
        [
            "hidden_layers",
            "A_train_kW",
            "A_validation_kW",
            "B_train_ms",
            "B_validation_ms",
            "C_train_kW",
            "C_validation_kW",
        ]
    ]
    .round(3)
    .to_string(index=False)
)


In [ ]:
# dropout test on the selected full-dataset architecture

def evaluate_dropout_full_dataset(
    dropout,
    hidden_layers=(64, 32),
    seed=SEED,
):
    rows = []

    targets = {
        "A": (
            train_data[["total_power"]].to_numpy(),
            validation_data[["total_power"]].to_numpy(),
            "kW",
        ),
        "B": (
            train_data[wind_speed_columns].to_numpy(),
            validation_data[wind_speed_columns].to_numpy(),
            "m/s",
        ),
        "C": (
            train_data[turbine_power_columns].to_numpy(),
            validation_data[turbine_power_columns].to_numpy(),
            "kW",
        ),
    }

    for model_name, (
        y_train_raw,
        y_validation_raw,
        unit,
    ) in targets.items():
        y_mean, y_std = fit_standardisation(
            y_train_raw
        )

        y_train = standardise(
            y_train_raw,
            y_mean,
            y_std,
        )

        output_size = y_train.shape[1]

        model = build_capacity_network(
            input_size=len(feature_columns),
            output_size=output_size,
            hidden_layers=hidden_layers,
            dropout=dropout,
        )

        model = train_fixed_epochs(
            model=model,
            X_train=X_train,
            y_train=y_train,
            epochs=CAPACITY_EPOCHS,
            learning_rate=CAPACITY_LEARNING_RATE,
            seed=seed,
        )

        with torch.no_grad():
            pred_train_raw = (
                model(
                    torch.tensor(
                        X_train,
                        dtype=torch.float32,
                    )
                ).numpy()
                * y_std
                + y_mean
            )

            pred_validation_raw = (
                model(
                    torch.tensor(
                        X_validation,
                        dtype=torch.float32,
                    )
                ).numpy()
                * y_std
                + y_mean
            )

        if model_name == "C":
            train_mae = mean_absolute_error(
                train_data["total_power"].to_numpy(),
                pred_train_raw.sum(axis=1),
            )

            validation_mae = mean_absolute_error(
                validation_data["total_power"].to_numpy(),
                pred_validation_raw.sum(axis=1),
            )
        else:
            train_mae = mean_absolute_error(
                y_train_raw.reshape(-1),
                pred_train_raw.reshape(-1),
            )

            validation_mae = mean_absolute_error(
                y_validation_raw.reshape(-1),
                pred_validation_raw.reshape(-1),
            )

        rows.append(
            {
                "model": model_name,
                "dropout": dropout,
                "unit": unit,
                "train_mae": train_mae,
                "validation_mae": validation_mae,
            }
        )

    return rows


dropout_rows = []

for dropout in DROPOUT_RATES:
    dropout_rows.extend(
        evaluate_dropout_full_dataset(
            dropout=dropout,
        )
    )

dropout_results = pd.DataFrame(
    dropout_rows
)

dropout_results.to_csv(
    CAPACITY_RESULTS_DIR / "dropout_full_dataset.csv",
    index=False,
)

print(
    dropout_results
    .round(3)
    .to_string(index=False)
)


In [ ]:
# positive-control experiment on 500 training timestamps

def run_positive_control(
    n_samples=500,
    hidden_layers=(1024, 512),
    dropout_rates=DROPOUT_RATES,
    epochs=3000,
    seed=42,
):
    set_seed(seed)

    rng = np.random.default_rng(seed)

    sample_indices = np.sort(
        rng.choice(
            len(train_data),
            size=n_samples,
            replace=False,
        )
    )

    sample_data = train_data.iloc[
        sample_indices
    ].copy()

    X_sample_raw = sample_data[
        feature_columns
    ].to_numpy()

    X_sample_mean, X_sample_std = fit_standardisation(
        X_sample_raw
    )

    X_sample = standardise(
        X_sample_raw,
        X_sample_mean,
        X_sample_std,
    )

    X_validation_sample_scale = standardise(
        X_validation_raw,
        X_sample_mean,
        X_sample_std,
    )

    rows = []

    target_sets = {
        "A": (
            sample_data[["total_power"]].to_numpy(),
            validation_data[["total_power"]].to_numpy(),
        ),
        "B": (
            sample_data[wind_speed_columns].to_numpy(),
            validation_data[wind_speed_columns].to_numpy(),
        ),
        "C": (
            sample_data[turbine_power_columns].to_numpy(),
            validation_data[turbine_power_columns].to_numpy(),
        ),
    }

    for dropout in dropout_rates:
        for model_name, (
            y_sample_raw,
            y_validation_raw,
        ) in target_sets.items():
            y_mean, y_std = fit_standardisation(
                y_sample_raw
            )

            y_sample = standardise(
                y_sample_raw,
                y_mean,
                y_std,
            )

            model = build_capacity_network(
                input_size=len(feature_columns),
                output_size=y_sample.shape[1],
                hidden_layers=hidden_layers,
                dropout=dropout,
            )

            model = train_fixed_epochs(
                model=model,
                X_train=X_sample,
                y_train=y_sample,
                epochs=epochs,
                learning_rate=CAPACITY_LEARNING_RATE,
                seed=seed,
            )

            with torch.no_grad():
                pred_sample = (
                    model(
                        torch.tensor(
                            X_sample,
                            dtype=torch.float32,
                        )
                    ).numpy()
                    * y_std
                    + y_mean
                )

                pred_validation = (
                    model(
                        torch.tensor(
                            X_validation_sample_scale,
                            dtype=torch.float32,
                        )
                    ).numpy()
                    * y_std
                    + y_mean
                )

            if model_name == "C":
                train_mae = mean_absolute_error(
                    sample_data["total_power"].to_numpy(),
                    pred_sample.sum(axis=1),
                )

                validation_mae = mean_absolute_error(
                    validation_data["total_power"].to_numpy(),
                    pred_validation.sum(axis=1),
                )
            else:
                train_mae = mean_absolute_error(
                    y_sample_raw.reshape(-1),
                    pred_sample.reshape(-1),
                )

                validation_mae = mean_absolute_error(
                    y_validation_raw.reshape(-1),
                    pred_validation.reshape(-1),
                )

            rows.append(
                {
                    "model": model_name,
                    "dropout": dropout,
                    "train_mae": train_mae,
                    "validation_mae": validation_mae,
                }
            )

    return pd.DataFrame(rows)


positive_control_results = run_positive_control()

positive_control_results.to_csv(
    CAPACITY_RESULTS_DIR / "positive_control_dropout.csv",
    index=False,
)

print(
    positive_control_results
    .round(3)
    .to_string(index=False)
)
